In [21]:
# ignoring hermit snail, only algae specimen
# tidy, long format of data
# only looking at reflectances between 400 nm and 1000 nm

import os
import glob
import re
import struct
import numpy as np
import pandas as pd


# 1. Exact Binary ASD Reader (Verified via Test A)
def read_as7_binary(filepath):
    """
    Decodes ASD FieldSpec Version 7 (as7) binary files.
    Reads 2151 channels as 64-bit doubles starting at offset 484.
    """
    with open(filepath, "rb") as f:
        data = f.read()

    channels = 2151
    # Physical standard wavelength array: 350 to 2500 nm (integer bands)
    wavelengths = np.arange(350, 350 + channels, dtype=int)

    # Offset 484, 64-bit doubles (<d)
    offset = 484
    values = struct.unpack(f"<{channels}d", data[offset : offset + channels * 8])

    s = pd.Series(values, index=wavelengths, dtype=float)
    s.name = os.path.splitext(os.path.basename(filepath))[0]
    return s


# 2. Main Traversal and Tidy Exporter
def export_tidy_reflectance_to_csv(
    root_dir, 
    output_filename="tidy_reflectance_algae_400_1000nm.csv",
    wl_min=400,
    wl_max=1000
):
    target_conditions = ["freshwater", "nowater", "saltwater"]
    records = []

    # Filter strictly for folders starting with "algae_"
    all_subdirs = [
        d for d in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, d)) and d.lower().startswith("algae_")
    ]

    def get_algae_num(name):
        match = re.search(r'\d+', name)
        return int(match.group()) if match else 9999

    specimen_folder_names = sorted(all_subdirs, key=get_algae_num)
    specimen_dirs = [os.path.join(root_dir, d) for d in specimen_folder_names]

    print(f"Targeting {len(specimen_dirs)} algae specimen folders:")
    print(f"{specimen_folder_names}\n")
    print(f"Slicing spectral bands: {wl_min} nm – {wl_max} nm\n")

    for spec_path in specimen_dirs:
        specimen_id = os.path.basename(spec_path)
        subdirs = {d.lower().strip(): d for d in os.listdir(spec_path) if os.path.isdir(os.path.join(spec_path, d))}

        # Match WR folder
        wr_folder_name = subdirs.get("wr")
        if not wr_folder_name:
            print(f"⚠️ Skipping '{specimen_id}': 'WR' folder not found.")
            continue

        wr_dir = os.path.join(spec_path, wr_folder_name)
        wr_files = sorted(glob.glob(os.path.join(wr_dir, "*.[aA][sS][dD]")))

        if not wr_files:
            print(f"⚠️ Skipping '{specimen_id}': No .asd files in {wr_dir}")
            continue

        # Load and average White Reference scans
        wr_series_list = []
        for wf in wr_files:
            try:
                wr_series_list.append(read_as7_binary(wf))
            except Exception as e:
                print(f"Error reading WR file {os.path.basename(wf)}: {e}")

        if not wr_series_list:
            continue

        mean_wr = pd.concat(wr_series_list, axis=1).mean(axis=1)

        specimen_sample_count = 0
        for cond_key in target_conditions:
            cond_folder_name = subdirs.get(cond_key)
            if not cond_folder_name:
                continue

            cond_dir = os.path.join(spec_path, cond_folder_name)
            sample_files = sorted(glob.glob(os.path.join(cond_dir, "*.[aA][sS][dD]")))

            for rep_idx, sf in enumerate(sample_files, start=1):
                try:
                    raw_s = read_as7_binary(sf)

                    # Compute reflectance: Sample DN / Specimen Mean WR DN
                    refl_values = raw_s.values / mean_wr.values
                    wl_array = np.array(raw_s.index)

                    # Filter strictly to 400 - 1000 nm
                    mask = (wl_array >= wl_min) & (wl_array <= wl_max)
                    filtered_wl = wl_array[mask]
                    filtered_refl = refl_values[mask]

                    temp_df = pd.DataFrame({
                        "wavelength_nm": filtered_wl,
                        "specimen": specimen_id,
                        "specimen_num": get_algae_num(specimen_id),
                        "species": "",  # Blank column ready for your manual labeling
                        "treatment": cond_key,
                        "replicate": rep_idx,
                        "reflectance": filtered_refl,
                        "original_file": os.path.basename(sf)
                    })

                    records.append(temp_df)
                    specimen_sample_count += 1

                except Exception as e:
                    print(f"Error reading {os.path.basename(sf)}: {e}")

        print(f"✓ {specimen_id}: Processed {specimen_sample_count} scans.")

    if not records:
        print("\n❌ No records were created.")
        return None

    # Merge all into one tidy DataFrame
    tidy_df = pd.concat(records, ignore_index=True)

    # Sort cleanly by specimen, treatment, replicate, and wavelength
    tidy_df.sort_values(by=["specimen_num", "treatment", "replicate", "wavelength_nm"], inplace=True)
    tidy_df.drop(columns=["specimen_num"], inplace=True)

    # Save to CSV
    output_path = os.path.join(root_dir, output_filename)
    tidy_df.to_csv(output_path, index=False)

    print(f"\n==========================================")
    print(f"✅ Successfully exported!")
    print(f"Saved to:           {output_path}")
    print(f"Total rows:         {len(tidy_df):,}")
    print(f"Algae specimens:    {tidy_df['specimen'].nunique()}")
    print(f"Wavelength bands:   {tidy_df['wavelength_nm'].min()} - {tidy_df['wavelength_nm'].max()} nm ({tidy_df['wavelength_nm'].nunique()} bands)")
    print(f"Treatments present: {tidy_df['treatment'].unique().tolist()}")
    print(f"==========================================")

    return tidy_df


# 3. Execution
root_data_dir = r"C:\Users\kimmi\OneDrive - Chapman University\SEACR\mkim_seacr_algaedata\asd_data\mkim_251121\mkim_251121"

tidy_df = export_tidy_reflectance_to_csv(root_data_dir, wl_min=400, wl_max=1000)

Targeting 24 algae specimen folders:
['algae_1', 'algae_2', 'algae_3', 'algae_4', 'algae_5', 'algae_6', 'algae_7', 'algae_8', 'algae_9', 'algae_10', 'algae_11', 'algae_12', 'algae_13', 'algae_14', 'algae_15', 'algae_16', 'algae_17', 'algae_18', 'algae_19', 'algae_20', 'algae_21', 'algae_22', 'algae_23', 'algae_24']

Slicing spectral bands: 400 nm – 1000 nm

✓ algae_1: Processed 15 scans.
✓ algae_2: Processed 15 scans.
✓ algae_3: Processed 15 scans.
✓ algae_4: Processed 15 scans.
✓ algae_5: Processed 15 scans.
✓ algae_6: Processed 15 scans.
✓ algae_7: Processed 15 scans.
✓ algae_8: Processed 15 scans.
✓ algae_9: Processed 15 scans.
✓ algae_10: Processed 15 scans.
✓ algae_11: Processed 15 scans.
✓ algae_12: Processed 15 scans.
✓ algae_13: Processed 15 scans.
✓ algae_14: Processed 15 scans.
✓ algae_15: Processed 15 scans.
✓ algae_16: Processed 15 scans.
✓ algae_17: Processed 15 scans.
✓ algae_18: Processed 15 scans.
✓ algae_19: Processed 15 scans.
✓ algae_20: Processed 15 scans.
✓ algae_2